<style>
  .nvidia-banner {background: linear-gradient(100deg,#0b0b0b,#292929); color:white;
                  border-left:10px solid #76b900; padding:18px 22px; margin:8px 0 18px;}
  .nvidia-banner h1 {margin:0 0 6px; font-size:30px;}
  .task {border-left:6px solid #76b900; background:#f5f8f1; padding:12px 16px; margin:12px 0;}
  .checkpoint {border:1px solid #b8b8b8; border-radius:6px; padding:10px 14px; background:#fafafa;}
  .warning {border-left:6px solid #f2a900; background:#fff8e6; padding:12px 16px;}
  code {font-size: 0.92em;}
</style>

<div class="nvidia-banner">
  <h1>Module 2 — Pair with a Coding Agent: Build a ReFRAME Neighborhood Atlas</h1>
  <div>ACS Fall 2026 · Agent-assisted scientific programming · 50–65 minutes</div>
</div>

## Goal

You now know the core calls. Your coding agent will help turn them into a reusable, tested function that asks a sharper question:

> **How stable are the nearest structural neighbors of imatinib, linezolid, and ritonavir when the Morgan radius changes?**

You remain the scientist: you define the contract, review the code, run tests, and interpret the result. The agent accelerates implementation.


## Setup

Use the same GPU environment as Module 1. Module 2 now calls the workshop's hosted LLM directly through `workshop_llm_agent.py`; no separate coding-agent application is required. You will explicitly enable generation, inspect the returned source, and let deterministic notebook tests decide whether it is usable.

The hosted call requires network access and an NVIDIA Developer API key (`nvapi-`). The reference implementation remains available when the embedded agent is disabled. Keep `workshop_llm_agent.py`, `workshop_common.py`, and `data/reframe_teaching_snapshot.csv` beside this notebook.


In [ ]:
# Prepare methods for comparing structural neighborhoods across fingerprint choices.
from pathlib import Path
import importlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from rdkit import DataStructs, RDLogger, rdBase
from rdkit.Chem import rdFingerprintGenerator

from workshop_common import add_descriptors, load_reframe

# Confirm that this notebook and its local workshop agent were released together.
import workshop_llm_agent as _workshop_llm_agent

EXPECTED_WORKSHOP_AGENT_VERSION = "2026.08.14.22"
_workshop_llm_agent = importlib.reload(_workshop_llm_agent)
loaded_agent_version = getattr(
    _workshop_llm_agent, "WORKSHOP_AGENT_VERSION", "pre-version"
)
if loaded_agent_version != EXPECTED_WORKSHOP_AGENT_VERSION:
    raise RuntimeError(
        "workshop_llm_agent.py is out of date: "
        f"expected {EXPECTED_WORKSHOP_AGENT_VERSION}, found {loaded_agent_version}. "
        "Replace the agent file with the copy distributed with this notebook, "
        "then restart the kernel and run this cell again."
    )
print(
    "Workshop agent:",
    loaded_agent_version,
    "|",
    _workshop_llm_agent.__file__,
)

RDLogger.DisableLog("rdApp.error")
SEED = 2026
np.random.seed(SEED)

# Use nvMolKit for batched fingerprint and similarity work when a GPU is available.
NVMOLKIT_READY = False
NVMOLKIT_IMPORT_ERROR = None
try:
    import torch
    import nvmolkit
    from nvmolkit.fingerprints import MorganFingerprintGenerator
    from nvmolkit.similarity import crossTanimotoSimilarity

    NVMOLKIT_READY = bool(torch.cuda.is_available())
except Exception as exc:
    NVMOLKIT_IMPORT_ERROR = repr(exc)

print(f"RDKit {rdBase.rdkitVersion}")
if NVMOLKIT_READY:
    print(f"nvMolKit {nvmolkit.__version__} | CUDA devices: {torch.cuda.device_count()}")
else:
    print("CPU teaching fallback active: nvMolKit GPU calls will be shown but evaluated with RDKit.")
    print("Reason:", NVMOLKIT_IMPORT_ERROR or "torch.cuda.is_available() is False")

In [ ]:
def fingerprint_tensor(result):
    """Return the CUDA torch tensor wrapped by an nvMolKit fingerprint result."""
    return result if isinstance(result, torch.Tensor) else result.torch()


def gpu_to_numpy(result):
    """Synchronize an nvMolKit result or CUDA tensor and return a host array."""
    if isinstance(result, torch.Tensor):
        return result.detach().cpu().numpy()
    return result.numpy()


# Morgan fingerprints describe local atomic environments at a chosen radius.
def make_fingerprints(molecules, radius=2, fp_bits=1024):
    """Return nvMolKit CUDA fingerprints, or RDKit fingerprints in fallback mode."""
    if NVMOLKIT_READY:
        generator = MorganFingerprintGenerator(radius=radius, fpSize=fp_bits)
        return fingerprint_tensor(generator.GetFingerprints(list(molecules), num_threads=0))
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_bits)
    return generator.GetFingerprints(list(molecules), numThreads=0)


# Tanimoto similarity compares the structural features encoded by two fingerprints.
def tanimoto_matrix(first, second=None):
    """Return a host similarity matrix from the active backend."""
    if NVMOLKIT_READY:
        return gpu_to_numpy(crossTanimotoSimilarity(first, second))
    second = first if second is None else second
    return np.asarray(
        [DataStructs.BulkTanimotoSimilarity(query, second) for query in first],
        dtype=float,
    )


print("Shared ReFRAME helpers and Module 2 fingerprint helpers are ready.")

### What the notebook computes

- **Morgan fingerprints** encode circular atom environments as a fixed-length bit vector. Radius changes how far each local environment extends; fingerprint length changes the collision budget.
- **Tanimoto similarity** compares two binary fingerprints. It is a structural-neighborhood measure, not a measurement of target binding or biological activity.
- **Butina clustering** groups molecules using a distance cutoff. Here, `distance = 1 - Tanimoto similarity`, so a cutoff of `0.55` corresponds to a similarity threshold of `0.45`.
- **ETKDG + MMFF94** samples and relaxes 3D conformers. Energies are only comparable among conformers of the same molecule under the same force-field setup.

The notebook uses nvMolKit when a compatible NVIDIA GPU is available. Its CPU fallback exists only so the teaching narrative and checks remain inspectable on a non-GPU laptop; the workshop's accelerated exercises should report `NVMOLKIT_READY == True`.


In [ ]:
# Hold the sample and reference compounds fixed so radius is the main comparison.
SAMPLE_SIZE = 10000
ANCHOR_TERMS = ("imatinib", "linezolid", "ritonavir")
RADII = (2, 3)
FP_BITS = 1024
TOP_K = 10

reframe = add_descriptors(load_reframe(SAMPLE_SIZE, anchor_terms=ANCHOR_TERMS))
print("Source:", reframe.attrs["source"])
print("Rows:", len(reframe), "| Backend:", "nvMolKit GPU" if NVMOLKIT_READY else "RDKit CPU fallback")


## Step 1 — Give the embedded agent a precise contract

A good coding-agent request includes the scientific question, real API signatures, output schema, constraints, and tests. The next cell constructs the exact prompt that the notebook will send to Nemotron. Read it before generating anything: you remain responsible for the contract.


In [ ]:
# Give the agent the scientific question, evidence standard, and interpretation limits.
AGENT_PROMPT = f"""
You are pairing with a chemist in an ACS workshop. Select two bounded
implementation policies. The local workshop agent will render them into the
tested `build_neighborhood_atlas` scaffold shown after your response.

Scientific question
-------------------
For {ANCHOR_TERMS}, compare the top {TOP_K} non-self structural neighbors in a
ReFRAME sample using Morgan radii {RADII} and {FP_BITS} bits.

Installed nvMolKit references
----------------------------
- Workshop skill: /nvmolkit-brev-notebook/skills/nvmolkit/SKILL.md
- Installed package: /.venv/lib/python3.12/site-packages/nvmolkit
- Repository: https://github.com/NVIDIA-BioNeMo/nvMolKit
- Documentation: https://nvidia-bionemo.github.io/nvMolKit/
Treat the installed skill and package as authoritative because this workshop uses
a newer nvMolKit build than the public repository.

Notebook helper contract
------------------------
- `make_fingerprints(molecules, radius=2, fp_bits=1024)` accepts one radius and
  returns fingerprints ready for `tanimoto_matrix`.
- `tanimoto_matrix(first, second=None)` returns a host NumPy array. Do not call
  `.numpy()` on its result.
- Do not add imports, access fingerprint internals, or call nvMolKit directly.

Concise implementation recipe
-----------------------------
1. Locate one row for each anchor with case-insensitive literal name matching.
2. Build the library molecule list and the query molecule list.
3. For each radius, fingerprint both lists and compute query-versus-library similarity.
4. Check the matrix shape and range, then use stable descending ranking.
5. Exclude the query by `canonical_ikey`, take exactly `top_k`, and append tidy rows.
6. Return the required columns sorted by radius, query, and rank.

Required scaffold
-----------------
- `anchor_terms` is a tuple. Use `for term in anchor_terms`; never call a string
  method on the tuple.
- Use these variables: `query_indices`, `molecules`, `query_molecules`, `rows`,
  `library_fps`, `query_fps`, `similarities`, and `columns`.
- Inside the radius loop, call exactly:
  `library_fps = make_fingerprints(molecules, radius=radius, fp_bits=fp_bits)`
  `query_fps = make_fingerprints(query_molecules, radius=radius, fp_bits=fp_bits)`
  `similarities = tanimoto_matrix(query_fps, library_fps)`
- Keep the full records table as the library. Filter self-matches only while ranking,
  so similarity columns remain aligned with `records.iloc` rows.
- Use `np.argsort(..., kind='stable')`. Populate names, connectivity keys, and
  `reframedb_url` from the matching records rows; never put molecule objects in output.

Bounded policy choices
----------------------
- `MISSING_ANCHOR`: choose `raise` for an explicit error or `skip` to continue with
  the anchors that were found.
- `INVALID_MATRIX`: choose `raise` for an explicit error or `skip` to omit that radius.
- Explain both choices briefly. Do not write Python; the local agent renders and
  validates the function so molecule rows and similarity columns cannot drift apart.

Function contract
-----------------
Inputs: records DataFrame with `_mol`, `name`, `canonical_ikey`, `reframedb_url`;
anchor_terms; radii; fp_bits; top_k.
Output: tidy DataFrame with exactly these columns:
radius, query, query_ikey, rank, neighbor, neighbor_ikey, tanimoto, profile.
Exclude the query itself by canonical_ikey. Sort by radius, query, rank.
Use the supplied `make_fingerprints` and `tanimoto_matrix` helpers so the same
function can be reviewed on a CPU-only laptop. Do not infer biological activity.

Quality bar
-----------
Add a one-sentence docstring, clear names, deterministic ranking, shape checks,
similarity range checks, and no duplicate neighbor within one (radius, query) group.
The local renderer uses a one-sentence docstring, no decorative padding, and at
most 55 physical lines so the displayed function remains below the hard 70-line cap.
Explain the two selected policies; the local checks validate the rendered Python.
""".strip()

display(Markdown("```text\n" + AGENT_PROMPT + "\n```"))


## Step 2 — Generate and inspect the proposed function

Set `RUN_EMBEDDED_AGENT = True`. The setup cell has already reloaded the local agent module and verified its version and path. This cell obtains a hidden API key, asks Nemotron to select and explain two bounded implementation policies, and renders those choices into a tested function scaffold. If the policy response is incomplete, the embedded agent receives the exact error and gets up to two bounded repair attempts. The rendered Python then passes the workshop's static contract checks before it is displayed.

The accepted source is displayed and cached, but **it is not executed**. Read it and confirm that it uses the supplied batch helpers, excludes self-matches by connectivity key, and makes no biological claims. Set `REGENERATE_AGENT_CODE = True` only when you intentionally want a new proposal.

<div class="warning"><b>Generated code is still untrusted.</b> Static checks reduce risk but are not a security sandbox. You decide whether to copy the displayed function into the exercise cell below.</div>


In [ ]:
# Ask the embedded agent to propose the structural-neighborhood analysis.
RUN_EMBEDDED_AGENT = True        # Generate a proposal in the hands-on exercise
REGENERATE_AGENT_CODE = True     # Request a fresh proposal for this run

agent_cache_dir = Path("module2_agent_workspace")
agent_cache_path = agent_cache_dir / "neighborhood_function.json"
generated_function_source = None
agent_design_choices = []

if RUN_EMBEDDED_AGENT:
    agent_api_key = _workshop_llm_agent.get_workshop_api_key()
    agent_cache_dir.mkdir(parents=True, exist_ok=True)
    if agent_cache_path.exists() and not REGENERATE_AGENT_CODE:
        generated_payload = json.loads(agent_cache_path.read_text(encoding="utf-8"))
        print("Loaded the cached embedded-agent proposal.")
    else:
        generated_result = _workshop_llm_agent.generate_neighborhood_function(
            AGENT_PROMPT, agent_api_key
        )
        generated_payload = generated_result.model_dump(mode="json")
        agent_cache_path.write_text(
            json.dumps(generated_payload, indent=2), encoding="utf-8"
        )
        print("Nemotron selected a method plan; the workshop agent rendered validated code.")

    generated_function_source = generated_payload["function_source"]
    agent_design_choices = generated_payload["design_choices"]
    display(Markdown("```python\n" + generated_function_source + "\n```"))
    display(Markdown("**Agent design choices**\n\n" + "\n".join(
        f"- {choice}" for choice in agent_design_choices
    )))
    print("The proposed function has been displayed but not executed.")
else:
    print("Embedded agent disabled; no function was generated or executed.")


## Step 3 — Review, copy, and define the function

1. Read the generated function displayed above.
2. Copy only the complete `build_neighborhood_atlas` function.
3. Replace the starter function in the next cell with the copied function.
4. Set `USE_REFERENCE_SOLUTION = False`.
5. Run the cell. This defines the function but does not yet perform the neighborhood analysis.

The following acceptance-test cell is where the pasted function is called on the ReFRAME sample.


In [ ]:
# Set False after replacing the starter with the agent's proposed method.
USE_REFERENCE_SOLUTION = True


# Paste the complete generated function below, replacing this starter.
def build_neighborhood_atlas(records, anchor_terms, radii=(2, 3), fp_bits=1024, top_k=10):
    """Starter placeholder; replace this function with the reviewed agent proposal."""
    return None


### Reference implementation

This tagged cell makes the notebook recoverable if the hosted service or participant implementation is unavailable. In a live workshop, leave it hidden until participants have reviewed their agent's implementation.


In [ ]:
# Build comparable neighbor lists at each radius using the same compounds.
def reference_build_neighborhood_atlas(records, anchor_terms, radii=(2, 3), fp_bits=1024, top_k=10):
    """Build a tidy, multi-radius structural-neighborhood atlas."""
    required = {'_mol', 'name', 'canonical_ikey', 'reframedb_url'}
    missing = required - set(records.columns)
    if missing:
        raise ValueError(f'records is missing {sorted(missing)}')
    query_indices = []
    for term in anchor_terms:
        matches = records[records['name'].str.contains(term, case=False, regex=False)]
        if matches.empty:
            raise ValueError(f'Anchor not found: {term}')
        query_indices.append(int(matches.index[0]))
    rows = []
    molecules = records['_mol'].tolist()
    query_molecules = records.loc[query_indices, '_mol'].tolist()
    # Each radius represents a different scale of local chemical context.
    for radius in radii:
        library_fps = make_fingerprints(molecules, radius=radius, fp_bits=fp_bits)
        query_fps = make_fingerprints(query_molecules, radius=radius, fp_bits=fp_bits)
        similarities = tanimoto_matrix(query_fps, library_fps)
        if similarities.shape != (len(query_indices), len(records)):
            raise RuntimeError(f'Unexpected similarity shape: {similarities.shape}')
        for query_position, query_index in enumerate(query_indices):
            query = records.loc[query_index]
            order = np.argsort(-similarities[query_position], kind='stable')
            # Exclude the reference itself so every reported match is a true neighbor.
            order = [int(idx) for idx in order if records.iloc[idx]['canonical_ikey'] != query['canonical_ikey']][:top_k]
            for rank, library_index in enumerate(order, start=1):
                neighbor = records.iloc[library_index]
                rows.append({'radius': int(radius), 'query': query['name'], 'query_ikey': query['canonical_ikey'], 'rank': rank, 'neighbor': neighbor['name'], 'neighbor_ikey': neighbor['canonical_ikey'], 'tanimoto': float(similarities[query_position, library_index]), 'profile': neighbor['reframedb_url']})
    columns = ['radius', 'query', 'query_ikey', 'rank', 'neighbor', 'neighbor_ikey', 'tanimoto', 'profile']
    return pd.DataFrame(rows, columns=columns).sort_values(['radius', 'query', 'rank'], ignore_index=True)

## Step 4 — Run the pasted function and its acceptance tests

This cell calls the function you defined in Step 3. The tests check structure and invariants; they cannot decide whether the scientific interpretation is sensible.


In [ ]:
# Test the result as scientific evidence, independent of who wrote the method.
atlas_builder = (
    reference_build_neighborhood_atlas
    if USE_REFERENCE_SOLUTION
    else build_neighborhood_atlas
)
print("Implementation under test:", atlas_builder.__name__)

atlas = atlas_builder(
    reframe, ANCHOR_TERMS, radii=RADII, fp_bits=FP_BITS, top_k=TOP_K
)

expected_columns = [
    "radius", "query", "query_ikey", "rank", "neighbor",
    "neighbor_ikey", "tanimoto", "profile"
]
assert list(atlas.columns) == expected_columns
assert len(atlas) == len(RADII) * len(ANCHOR_TERMS) * TOP_K
assert atlas["tanimoto"].between(0, 1).all()
assert (atlas["query_ikey"] != atlas["neighbor_ikey"]).all()
assert atlas.groupby(["radius", "query"])["neighbor_ikey"].nunique().eq(TOP_K).all()
assert atlas.groupby(["radius", "query"])["rank"].apply(list).apply(lambda x: x == list(range(1, TOP_K + 1))).all()
print("✓ The selected neighborhood-atlas implementation passes all acceptance tests.")
display(atlas.round({"tanimoto": 3}).head(12))


## Step 5 — Compare the methods, not just the molecules

We use Jaccard overlap of the two top-10 sets. `1.0` means identical membership; `0.0` means no shared neighbors. Rank order can still change even when membership is identical.


In [ ]:
# Jaccard overlap shows how much each neighbor set changes with fingerprint radius.
overlap_rows = []
radius_a, radius_b = RADII
for query in atlas["query"].unique():
    set_a = set(atlas.query("radius == @radius_a and query == @query")["neighbor_ikey"])
    set_b = set(atlas.query("radius == @radius_b and query == @query")["neighbor_ikey"])
    overlap_rows.append({
        "query": query,
        f"radius_{radius_a}_only": len(set_a - set_b),
        "shared": len(set_a & set_b),
        f"radius_{radius_b}_only": len(set_b - set_a),
        "jaccard": len(set_a & set_b) / len(set_a | set_b),
    })

overlap = pd.DataFrame(overlap_rows).sort_values("jaccard")
display(overlap.round(3))
overlap.plot.bar(x="query", y="jaccard", ylim=(0, 1), color="#76b900", legend=False, figsize=(8, 3.5))
plt.ylabel("Top-10 Jaccard overlap")
plt.title("Neighborhood stability when Morgan radius changes")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## Step 6 — Ask the embedded agent to review, not merely extend

When generation is enabled, the next cell sends the implementation and bounded result summary back to the same in-notebook LLM. Its response is schema-checked into three distinct categories: a software failure mode, a representation limitation, and an unsupported biological inference. It also proposes three tests.

Review the suggestions and add **one** useful test in the following participant cell. A useful test should fail for a concrete defect, not merely restate an assertion already present.


In [ ]:
# Ask the agent to separate method limitations from biological interpretation.
agent_review = None
if RUN_EMBEDDED_AGENT and not USE_REFERENCE_SOLUTION:
    review_cache_path = agent_cache_dir / "neighborhood_review.json"
    if review_cache_path.exists() and not REGENERATE_AGENT_CODE:
        review_payload = json.loads(review_cache_path.read_text(encoding="utf-8"))
        print("Loaded the cached embedded-agent review.")
    else:
        result_summary = overlap.round(4).to_json(orient="records")
        review_result = _workshop_llm_agent.review_neighborhood_function(
            prompt=AGENT_PROMPT,
            function_source=generated_function_source,
            result_summary=result_summary,
            api_key=agent_api_key,
        )
        review_payload = review_result.model_dump(mode="json")
        review_cache_path.write_text(
            json.dumps(review_payload, indent=2), encoding="utf-8"
        )
    review_lines = [
        f"- **Software failure mode:** {review_payload['software_failure_mode']}",
        f"- **Representation limitation:** {review_payload['representation_limitation']}",
        f"- **Unsupported inference:** {review_payload['unsupported_biological_inference']}",
        "- **Proposed tests:**",
        *(f"  {index}. {test}" for index, test in enumerate(review_payload["proposed_tests"], 1)),
    ]
    display(Markdown("\n".join(review_lines)))
else:
    print("Embedded review skipped because the generated function was not used.")


In [ ]:
# Participant test: confirm each neighborhood runs from most to least similar.
monotonic = atlas.groupby(["radius", "query"])["tanimoto"].apply(
    lambda values: values.is_monotonic_decreasing
)
assert monotonic.all()
print("✓ Review test passed: every neighborhood is sorted by descending similarity.")


## Checks and next step

Discuss with a partner:

1. Did the agent honor the batch API, or accidentally loop over nvMolKit one molecule at a time?
2. Which anchor was most sensitive to radius, and what structural feature might explain that?
3. Which part required more scientific judgment: writing the function or interpreting its output?

In Module 3, you will stop specifying the implementation and instead give a full coding agent an objective, boundaries, and acceptance tests.


## Sources and scientific boundary

- [nvMolKit repository](https://github.com/NVIDIA-BioNeMo/nvMolKit)
- [nvMolKit documentation](https://nvidia-bionemo.github.io/nvMolKit/)
- [Installed workshop nvMolKit skill](../skills/nvmolkit/SKILL.md) — authoritative for this environment
- [reframeDb](https://reframedb.org/) and its public `reframe_smiles_list.csv` export

The ReFRAME export is used for teaching and should be handled under the site's current terms. Refresh it before delivery and do not treat availability status as evidence of clinical suitability. Fingerprints, descriptors, clusters, and sampled force-field geometries do **not** establish binding, activity, ADMET, efficacy, safety, synthesizability, or experimental structure.


In [ ]:
# The lowest overlap marks the neighborhood most sensitive to representation choice.
answer_sensitivity = overlap.sort_values(["jaccard", "query"], ascending=[True, True]).reset_index(drop=True)
print("Most radius-sensitive anchor = lowest top-10 Jaccard overlap")
display(answer_sensitivity.round(3))

most_sensitive_query = answer_sensitivity.iloc[0]["query"]
most_sensitive_jaccard = float(answer_sensitivity.iloc[0]["jaccard"])
print(f"Answer for this run: {most_sensitive_query} (Jaccard = {most_sensitive_jaccard:.3f})")


## Answer key — discussion checkpoint

<details>
<summary><b>Reveal instructor key</b></summary>

1. **Did the agent honor the batch API?** The displayed source and the function pasted into Step 3—not the agent's explanation—are the receipts. The embedded controller permits only one function and the notebook acceptance tests check its outputs, but participants must still inspect whether the implementation batches work. The reference implementation does. For each radius it makes one library fingerprint batch, one query fingerprint batch, and one cross-Tanimoto call; it does not call nvMolKit separately for each molecule. With two radii, that is four batched fingerprint calls and two batched similarity calls. For participant code, inspect the implementation: a Python loop over molecules that invokes nvMolKit per molecule is not full credit, even if the output passes.

2. **Most radius-sensitive anchor:** The correct answer for the current run is the first row of `answer_sensitivity`, which has the lowest top-10 Jaccard overlap. A plausible explanation must reference actual structural differences among the moved neighbors. Radius 3 incorporates larger connected environments than radius 2; therefore scaffold context and substitution patterns can displace molecules that only share smaller local motifs. Do not award full credit for merely saying “radius 3 is more accurate”—it is a different representation, not universally more correct.

3. **Where was scientific judgment required?** Interpreting the output required more scientific judgment than implementing the bounded function. The function contract and invariant tests determine software correctness; they cannot decide whether radius 2 or 3 is appropriate, whether a neighborhood shift is chemically meaningful, or what biological experiment follows. A strong answer separates code correctness from representation choice and biological inference.

**Review-test key:** The supplied monotonicity test is useful because each neighborhood must be ordered by decreasing similarity. Additional strong tests include deterministic tie handling, missing-anchor behavior, duplicate connectivity keys, and agreement with a small independently computed RDKit example.

</details>
